In [ ]:
# Pastikan PyTorch stack cocok dengan CUDA 12.6 bawaan Colab
!pip install -U "torch==2.8.0+cu126" "torchvision==0.20.0+cu126" "torchaudio==2.8.0+cu126" \
  -f https://download.pytorch.org/whl/cu126

# Balikkan paket inti ke versi yang aman buat Colab + YOLOv7
!pip install -U \
  "numpy>=2,<2.3" \
  "pandas==2.2.2" \
  "requests==2.32.4" \
  "opencv-python-headless==4.12.0.88" \
  "matplotlib==3.10.0" \
  "seaborn==0.13.2" \
  "pycocotools==2.0.7" \
  "thop==0.1.1.post2209072238"


Looking in links: https://download.pytorch.org/whl/cu126
ERROR: Ignored the following yanked versions: 0.1.6, 0.1.7, 0.1.8, 0.1.9, 0.2.0, 0.2.1, 0.2.2, 0.2.2.post2, 0.2.2.post3
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.0+cu126 (from versions: 0.17.0, 0.17.1, 0.17.2, 0.18.0, 0.18.1, 0.19.0, 0.19.1, 0.20.0, 0.20.1, 0.21.0, 0.22.0, 0.22.1, 0.23.0)
ERROR: No matching distribution found for torchvision==0.20.0+cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 126.5 MB/s eta 0:00:00
  Created wheel for pycocotools: filename=pycocotools-2.0.7-cp312-cp312-linux_x86_64.whl size=435632 sha256=3cb859cfa25d5109e14713d3b2136b40a318bf4e5d0e7be17d13993158b1ceb7
  Stored in directory: /root/.cache/pip/wheels/1e/b8/6d/646852bb348f96f4928351

In [ ]:
%cd /content
!rm -rf yolov7
!git clone https://github.com/WongKinYiu/yolov7
%cd yolov7

# (opsional) kalau mau install, gunakan no-build-isolation supaya pakai numpy yg sudah terpasang
# !pip install -r requirements.txt --no-build-isolation --no-cache-dir


/content
Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.29 MiB | 36.09 MiB/s, done.
Resolving deltas: 100% (511/511), done.
/content/yolov7


In [ ]:
!pip install -U roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="fWuJ46sQNxgW5o508yPv")
project = rf.workspace("test-ikan").project("ikanikan2")
version = project.version(5)
dataset = version.download("yolov7")
DATA_YAML = dataset.location + "/data.yaml"
print(DATA_YAML)


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ikanikan2-5 in yolov7pytorch:: 100%|██████████| 6966/6966 [00:00<00:00, 8365.66it/s]

/content/yolov7/ikanikan2-5/data.yaml


In [ ]:
%cd /content/yolov7
# paksa torch.load pakai weights_only=False
!sed -i "s/torch.load(weights, map_location=device)/torch.load(weights, map_location=device, weights_only=False)/" train.py


/content/yolov7


In [ ]:
!python train.py \
  --workers 2 \
  --device 0 \
  --batch-size 16 \
  --img-size 640 \
  --cfg cfg/training/yolov7-tiny.yaml \
  --weights yolov7-tiny.pt \
  --data "$DATA_YAML" \
  --name ikan_yolov7_tiny \
  --hyp data/hyp.scratch.tiny.yaml \
  --epochs 100 \
  --save_period 5


2025-09-24 14:45:40.604417: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758725140.624017    2901 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758725140.629919    2901 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758725140.644950    2901 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758725140.644976    2901 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758725140.644980    2901 computation_placer.cc:177] computation placer alr

In [ ]:
import glob, os
from google.colab import files

candidates = sorted(
    glob.glob("/content/yolov7/runs/train/*/weights/best.pt"),
    key=os.path.getmtime
)
assert candidates, "Belum ada best.pt di runs/train/*/weights/"
BEST_PATH = candidates[-1]
print("Best terbaru:", BEST_PATH)
files.download(BEST_PATH)


In [ ]:
# Val (mAP/PR)
!python test.py \
  --data "$DATA_YAML" \
  --img 640 \
  --batch 16 \
  --conf 0.001 \
  --device 0 \
  --weights runs/train/ikan_yolov7_tiny/weights/best.pt \
  --task val

# Deteksi contoh pada folder valid images
!python detect.py \
  --weights runs/train/ikan_yolov7_tiny/weights/best.pt \
  --conf 0.25 \
  --img-size 640 \
  --source {dataset.location + "/valid/images"} \
  --save-txt --save-conf
